## 1. CONEXIÓN CON LA BASE DE DATOS

## 1.1 Configuramos las credenciales y la contraseña oculta

In [5]:
import pandas as pd
from sqlalchemy import create_engine
import getpass 
from urllib.parse import quote_plus


# 1. Configuramos las credenciales del servidor local
USUARIO = 'postgres'
password = getpass.getpass('Introduce tu contraseña de pgAdmin: ') 
CONTRASENA = quote_plus(password)
HOST = 'localhost'
PUERTO = '5432'
BASE_DATOS = 'metrobus_db'

# 2. Creamos la conexion a la base de datos PostgreSQL usando SQLAlchemy
conexion = f"postgresql+psycopg2://{USUARIO}:{CONTRASENA}@{HOST}:{PUERTO}/{BASE_DATOS}"

# 3. Encendemos el motor
engine = create_engine(conexion)

print("Se ha conectado correctamente a la base de datos PostgreSQL.")

Se ha conectado correctamente a la base de datos PostgreSQL.


## 1.2 Inyección de datos

In [6]:
# --- INYECCIÓN DE DATOS: NIVEL 1 (DIMENSIONES INDEPENDIENTES) ---

# 1. Cargamos los CSV limpios en DataFrames de Pandas
df_tarifa = pd.read_csv('data/dim_tarifa.csv')
df_depot = pd.read_csv('data/dim_depot.csv')
df_linea = pd.read_csv('data/dim_linea.csv')
df_parada = pd.read_csv('data/dim_parada.csv')

print("Iniciando el volcado del Nivel 1 hacia PostgreSQL...")

# 2. Enviamos los datos a través del motor
df_tarifa.to_sql(name='dim_tarifa', con=engine, if_exists='append', index=False)
print("Tabla 'dim_tarifa' inyectada con éxito.")

df_depot.to_sql(name='dim_depot', con=engine, if_exists='append', index=False)
print("Tabla 'dim_depot' inyectada con éxito.")

df_linea.to_sql(name='dim_linea', con=engine, if_exists='append', index=False)
print("Tabla 'dim_linea' inyectada con éxito.")

df_parada.to_sql(name='dim_parada', con=engine, if_exists='append', index=False)
print("Tabla 'dim_parada' inyectada con éxito.")

print("Dimensiones independientes inyectadas correctamente en PostgreSQL.")

Iniciando el volcado del Nivel 1 hacia PostgreSQL...


DatabaseError: Execution failed on sql 'INSERT INTO dim_tarifa (tarifa_id, tipo_titulo, categoria, precio_eur, es_abono, bonificado) VALUES (:tarifa_id, :tipo_titulo, :categoria, :precio_eur, :es_abono, :bonificado)': (psycopg2.errors.UniqueViolation) duplicate key value violates unique constraint "dim_tarifa_pkey"
DETAIL:  Key (tarifa_id)=(1) already exists.

[SQL: INSERT INTO dim_tarifa (tarifa_id, tipo_titulo, categoria, precio_eur, es_abono, bonificado) VALUES (%(tarifa_id__0)s, %(tipo_titulo__0)s, %(categoria__0)s, %(precio_eur__0)s, %(es_abono__0)s, %(bonificado__0)s), (%(tarifa_id__1)s, %(tipo_titulo__1)s ... 765 characters truncated ... d__8)s, %(tipo_titulo__8)s, %(categoria__8)s, %(precio_eur__8)s, %(es_abono__8)s, %(bonificado__8)s)]
[parameters: {'tipo_titulo__0': 'Ordinario', 'precio_eur__0': 1.5, 'es_abono__0': False, 'categoria__0': 'Adulto', 'tarifa_id__0': 1, 'bonificado__0': False, 'tipo_titulo__1': 'Bono 10 Viajes', 'precio_eur__1': 0.97, 'es_abono__1': True, 'catego

In [7]:
# --- INYECCIÓN DE DATOS: NIVEL 2 (DIMENSIONES DEPENDIENTES) ---

# 1. Cargamos los archivos de las últimas dimensiones las cuales son dependientes de otras
df_conductor = pd.read_csv('data/dim_conductor.csv')
df_vehiculo = pd.read_csv('data/dim_vehiculo.csv')

print("Iniciando inyección de las últimas dimensiones...")

# 2. Inyectamos los conductores
df_conductor.to_sql(name='dim_conductor', con=engine, if_exists='append', index=False)
print("Tabla 'dim_conductor' inyectada con éxito.")

# 3. Inyectamos los vehículos
df_vehiculo.to_sql(name='dim_vehiculo', con=engine, if_exists='append', index=False)
print("Tabla 'dim_vehiculo' inyectada con éxito.")

Iniciando inyección de las últimas dimensiones...


DatabaseError: Execution failed on sql 'INSERT INTO dim_conductor (conductor_id, nombre, anno_incorporacion, antiguedad_anos, turno_habitual, depot_id, formacion, licencia_tipo, activo, ausencias_2024) VALUES (:conductor_id, :nombre, :anno_incorporacion, :antiguedad_anos, :turno_habitual, :depot_id, :formacion, :licencia_tipo, :activo, :ausencias_2024)': (psycopg2.errors.UndefinedColumn) column "turno_habitual" of relation "dim_conductor" does not exist
LINE 1: ..._id, nombre, anno_incorporacion, antiguedad_anos, turno_habi...
                                                             ^

[SQL: INSERT INTO dim_conductor (conductor_id, nombre, anno_incorporacion, antiguedad_anos, turno_habitual, depot_id, formacion, licencia_tipo, activo, ausencias_2024) VALUES (%(conductor_id__0)s, %(nombre__0)s, %(anno_incorporacion__0)s, %(antiguedad_anos ... 6227 characters truncated ...  %(depot_id__29)s, %(formacion__29)s, %(licencia_tipo__29)s, %(activo__29)s, %(ausencias_2024__29)s)]
[parameter